# T-163 arrival breaks validation — Phase 2 smoke

Post-Phase-1 validation for PR #65 / T-163. **Not** the nightly guard suite.

Covers: v2 artifact schema, fast session init (PR #71 thermal cache), multilot L=3, ρ=0 trace smoke, freshness + Prior calibration.

Prerequisites:

```bash
uv sync --extra dev --extra notebooks --extra rust --python 3.11
uv run maturin develop --release --manifest-path crates/voi_py/Cargo.toml
export BLUEBERRIES_VOI_BACKEND=rust
```

Target runtime: **< 10 minutes**.

In [1]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd().resolve()
if not (REPO / "src" / "blueberries_voi").is_dir():
    REPO = REPO.parent
assert (REPO / "data" / "abdella" / "arrival_model.json").is_file(), REPO

os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")

from blueberries_voi.backend import rust_available, warn_fallback_once

warn_fallback_once()
print("repo:", REPO)
print("rust backend:", rust_available())

repo: /home/oliver/blog/blueberries-voi
rust backend: True


## 1 — v2 artifact smoke (S1.9)

In [2]:
artifact_path = REPO / "data" / "abdella" / "arrival_model.json"
artifact = json.loads(artifact_path.read_text())

required = ("legs", "thermal_modes", "sigma_hour", "T_break", "rho", "tau_bar")
for key in required:
    assert key in artifact, f"missing {key}"
for gone in ("mu_T", "sigma_T", "temp_floor_c"):
    assert gone not in artifact, f"retired key {gone} still present"
assert "abdella_all" in artifact["corridors"]

ref_life = artifact["reference_life_days"]
print(f"reference_life_days={ref_life}  sigma_hour={artifact['sigma_hour']}  rho={artifact['rho']}")
assert ref_life == 26.0, f"Phase 1 expects reference_life_days=26, got {ref_life}"

pd.DataFrame(
    [
        {"check": "v2 schema keys", "status": "pass"},
        {"check": "retired keys absent", "status": "pass"},
        {"check": "unified abdella_all corridor", "status": "pass"},
        {"check": "reference_life_days=26", "status": "pass"},
    ]
)

reference_life_days=26.0  sigma_hour=0.028  rho=0.08


,check,status
0,v2 schema keys,pass
1,retired keys absent,pass
2,unified abdella_all corridor,pass
3,reference_life_days=26,pass


## 2 — Fast Rust tests (v2 generative, multilot, freshness)

In [3]:
def cargo_test(test_file: str, *extra: str) -> str:
    cmd = [
        "cargo",
        "test",
        "-p",
        "voi_core",
        "--release",
        "--test",
        test_file,
        "--",
        *extra,
    ]
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, check=True)
    elapsed = time.perf_counter() - t0
    print(f"{test_file}: {elapsed:.1f}s")
    return proc.stdout + proc.stderr


results: list[dict[str, str | float]] = []
for test_file, label in [
    ("t163_v2_artifact", "S1.9 artifact"),
    ("t163_freshness_calibration", "Phase 1 freshness"),
]:
    t0 = time.perf_counter()
    out = cargo_test(test_file)
    elapsed = time.perf_counter() - t0
    passed = "test result: ok" in out
    results.append({"suite": label, "seconds": round(elapsed, 1), "status": "pass" if passed else "fail"})
    assert passed, out[-500:]

pd.DataFrame(results)

t163_v2_artifact: 6.6s


t163_freshness_calibration: 14.6s


,suite,seconds,status
0,S1.9 artifact,6.6,pass
1,Phase 1 freshness,14.6,pass


## 3 — ρ=0 trace smoke (S1.4: hourly OU, calendar d)

In [4]:
out = cargo_test("t163_v2_generative", "rho_zero_trace_has_hourly_ou_variation", "--exact")
assert "test result: ok" in out
print("ρ=0 trace has hourly OU variation — pass")

t163_v2_generative: 0.1s
ρ=0 trace has hourly OU variation — pass


## 4 — Freshness distribution + belief (Phase 1)

In [5]:
def cargo_example(name: str) -> str:
    cmd = ["cargo", "run", "-p", "voi_core", "--release", "--example", name]
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, check=True)
    elapsed = time.perf_counter() - t0
    print(f"{name}: {elapsed:.1f}s\n{proc.stdout}")
    return proc.stdout


f_out = cargo_example("t163_f_diag")
session_out = cargo_example("t163_session_f_diag")

# Parse key metrics from stdout
p50 = float(re.search(r"p50=([0-9.]+)", f_out).group(1))
pct_below = float(re.search(r"pct<f0\.5=([0-9.]+)%", f_out).group(1))
prior_mean = float(re.search(r"Filter Prior mean_f=([0-9.]+)", f_out).group(1))
delivery_mean = float(re.search(r"delivery weighted mean_f=([0-9.]+)", session_out).group(1))
n_lots = int(re.search(r"n_lots=(\d+)", session_out).group(1))

metrics = pd.DataFrame(
    [
        {"metric": "arrival f p50", "value": p50, "gate": "≥ 0.65"},
        {"metric": "pct f < 0.5", "value": pct_below, "gate": "≪ pre-calibration mass"},
        {"metric": "session delivery mean_f", "value": delivery_mean, "gate": "≥ 0.55"},
        {"metric": "Prior mean_f", "value": prior_mean, "gate": "tracks multilot truth"},
        {"metric": "multilot lots per delivery", "value": n_lots, "gate": "3"},
    ]
)
metrics

t163_f_diag: 11.9s
Truth arrival f: mean=0.734 p10=0.566 p50=0.745 p90=0.894
  pct<f0.5=4.8%  pct in [0.6,0.9]=76.1%
  mean lambda=6.899
Filter Prior mean_f=0.685 sd=0.145
Filter Prior sample mean_f=0.683
Belief bias (filter-truth) at Prior: -0.052



t163_session_f_diag: 12.0s
day=1 arrivals=64 n_lots=3
  lot_id=3 n=21 mean_f=0.599
  lot_id=2 n=21 mean_f=0.725
  lot_id=1 n=22 mean_f=0.685
  delivery weighted mean_f=0.670
  WRONG last-lot-padded mean=0.197



,metric,value,gate
0,arrival f p50,0.745,≥ 0.65
1,pct f < 0.5,4.800,≪ pre-calibration mass
2,session delivery mean_f,0.670,≥ 0.55
3,Prior mean_f,0.685,tracks multilot truth
4,multilot lots per delivery,3.000,3


In [6]:
assert p50 >= 0.65
assert pct_below < 20.0
assert delivery_mean >= 0.55
assert n_lots == 3
print("Phase 1 freshness gates: pass")

Phase 1 freshness gates: pass


## 5 — Fast session init (PyO3, post thermal cache)

In [7]:
if rust_available():
    from blueberries_voi.controller.session_loop import default_session_config
    from blueberries_voi.simulator import EngineSession

    t0 = time.perf_counter()
    session = EngineSession()
    session.init(default_session_config(n_particles=200, obs_scenario="P0"), seed=42)
    init_ms = (time.perf_counter() - t0) * 1000
    print(f"EngineSession.init: {init_ms:.0f} ms")
    assert init_ms < 30_000, "init should not be pathological (>30s)"
else:
    print("SKIP: _core not built — run maturin develop --release")

EngineSession.init: 12961 ms


## 6 — Summary

| Check | Status |
|-------|--------|
| S1.9 v2 artifact schema | pass |
| S1.4 ρ=0 OU traces | pass |
| S2.1 multilot L=3 session | pass |
| Phase 1 freshness band | pass |
| Prior multilot bias ≤ 0.03 | pass (Rust test) |
| Fast session init | pass |

**Known follow-up:** ladder MAE(P0)/MAE(F2) ≈ 1.07 (target ≥ 3.0) — not weakened; tracked for later tuning.